In [21]:
!pip install pgmpy scikit-learn pandas

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

data = pd.read_csv("heart.csv")

print("Heart Disease Dataset:")
print(data.head())

print("\nDataset Shape:", data.shape)

print("\nColumn Names:")
print(data.columns.tolist())

Heart Disease Dataset:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   52    1   0       125   212    0        1      168      0      1.0      2   
1   53    1   0       140   203    1        0      155      1      3.1      0   
2   70    1   0       145   174    0        1      125      1      2.6      0   
3   61    1   0       148   203    0        1      161      0      0.0      2   
4   62    0   0       138   294    1        1      106      0      1.9      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  

Dataset Shape: (1025, 14)

Column Names:
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [23]:
continuous_columns = ["age", "trestbps", "chol", "thalach", "oldpeak"]

for column in continuous_columns:
    data[column] = pd.qcut(
        data[column],
        q=3,
        labels=False,
        duplicates="drop"
    )

data = data.astype(int)

print("Processed Dataset:")
print(data.head())

print("\nProcessed Dataset Shape:", data.shape)

Processed Dataset:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0    1    1   0         1     0    0        1        2      0        1      2   
1    1    1   0         2     0    1        0        1      1        2      0   
2    2    1   0         2     0    0        1        0      1        2      0   
3    2    1   0         2     0    0        1        1      0        0      2   
4    2    0   0         1     2    1        1        0      0        2      1   

   ca  thal  target  
0   2     3       0  
1   0     3       0  
2   0     3       0  
3   1     3       0  
4   3     2       0  

Processed Dataset Shape: (1025, 14)


In [24]:
train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["target"]
)

print("Training Data:", train_data.shape)
print("Testing Data:", test_data.shape)

Training Data: (820, 14)
Testing Data: (205, 14)


In [25]:
model = DiscreteBayesianNetwork([
    ("age", "target"),
    ("sex", "target"),
    ("cp", "target"),
    ("trestbps", "target"),
    ("chol", "target"),
    ("fbs", "target"),
    ("restecg", "target"),
    ("thalach", "target"),
    ("exang", "target"),
    ("oldpeak", "target"),
    ("slope", "target"),
    ("ca", "target"),
    ("thal", "target")
])

print("Bayesian Network Structure:")
print(model.edges())

Bayesian Network Structure:
[('age', 'target'), ('sex', 'target'), ('cp', 'target'), ('trestbps', 'target'), ('chol', 'target'), ('fbs', 'target'), ('restecg', 'target'), ('thalach', 'target'), ('exang', 'target'), ('oldpeak', 'target'), ('slope', 'target'), ('ca', 'target'), ('thal', 'target')]


In [26]:
from pgmpy.estimators import MaximumLikelihoodEstimator

estimator = MaximumLikelihoodEstimator(model, train_data)

cpds = estimator.get_parameters()

model.add_cpds(*cpds)

print("Bayesian Network trained successfully.")
print("\nNumber of CPDs:", len(model.get_cpds()))

/tmp/ipykernel_35155/3863938024.py:3: FutureWarning: `pgmpy.estimators.MaximumLikelihoodEstimator` is deprecated and will be removed in v1.3.0. Please use `pgmpy.parameter_estimator.DiscreteMLE` instead.
  estimator = MaximumLikelihoodEstimator(model, train_data)


Bayesian Network trained successfully.

Number of CPDs: 14


In [27]:
print("Model Check:", model.check_model())

Model Check: True


In [28]:
from pgmpy.inference import VariableElimination

inference = VariableElimination(model)

print("Starting diagnosis...")

predictions = []

for _, row in test_data.iterrows():
    evidence = row.drop("target").to_dict()
    result = inference.map_query(
        variables=["target"],
        evidence=evidence,
        show_progress=False
    )
    predictions.append(result["target"])

print("Diagnosis completed!")

print("\nActual Values:")
print(test_data["target"].values)

print("\nPredicted Values:")
print(predictions)

Starting diagnosis...
Diagnosis completed!

Actual Values:
[0 1 0 1 0 0 1 0 1 1 0 1 0 0 1 0 0 0 0 1 0 0 1 1 0 1 0 0 1 0 1 1 0 1 0 0 1
 0 0 1 0 1 1 1 0 0 1 1 0 0 0 0 1 0 1 0 1 1 0 0 1 1 1 0 1 0 0 1 1 1 1 0 1 1
 0 0 1 0 0 0 1 1 1 1 1 0 1 1 1 0 1 1 1 0 1 1 1 1 0 1 1 1 1 1 1 1 1 0 0 1 1
 0 1 1 1 0 1 1 1 1 1 0 0 1 1 0 0 0 0 0 0 0 1 0 0 1 0 1 0 1 1 1 0 0 0 0 1 1
 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 1 1 0 0 0 1 1 1 1 0 1 0 1 1 0 1 1 0
 1 0 0 0 1 0 1 1 0 1 0 1 1 0 1 0 0 0 1 1]

Predicted Values:
[np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0

In [29]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(test_data["target"], predictions)
precision = precision_score(test_data["target"], predictions, zero_division=0)
recall = recall_score(test_data["target"], predictions, zero_division=0)

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall:", round(recall * 100, 2), "%")

Accuracy: 98.54 %
Precision: 100.0 %
Recall: 97.14 %


In [30]:
patient = test_data.drop(columns=["target"]).iloc[0].to_dict()

diagnosis = inference.map_query(
    variables=["target"],
    evidence=patient,
    show_progress=False
)

print("Patient Medical Data:")
print(patient)

print("\nPredicted Diagnosis:", diagnosis["target"])

if diagnosis["target"] == 1:
    print("Diagnosis: Heart Disease Present")
else:
    print("Diagnosis: Heart Disease Absent")

Patient Medical Data:
{'age': 2, 'sex': 1, 'cp': 0, 'trestbps': 0, 'chol': 1, 'fbs': 0, 'restecg': 0, 'thalach': 0, 'exang': 1, 'oldpeak': 2, 'slope': 0, 'ca': 1, 'thal': 2}

Predicted Diagnosis: 0
Diagnosis: Heart Disease Absent
